<div style="background: linear-gradient(135deg, #0d1117 0%, #161b22 50%, #0d1117 100%); border: 2px solid #30363d; border-radius: 12px; padding: 30px; margin-bottom: 20px;">

<h1 style="color: #f0f6fc; text-align: center; font-size: 2em; margin: 0 0 12px 0;">
  Privacy Score 정량화 실험 - SSIM 기반, PSNR 참고 지표
</h1>

<p style="color: #8b949e; text-align: center; margin: 0;">
블록 픽셀 크기별 / 가우시안 블러별 / 블록+블러 조합별로 원본 대비 구조적 차이를 계산하고<br>
SSIM 기반 Privacy Score를 표와 상세 그래프로 저장합니다.
</p>

</div>


## 사용한 Privacy Score 수식

이 노트북의 최종 privacy score는 **MS-SSIM(Multi-Scale SSIM)** 을 사용합니다. 블록/블러/조합
세 가지 변환을 같은 수식으로, 최대한 공정하게 비교하기 위한 선택입니다. PSNR과 단일 스케일
SSIM은 점수 계산에 넣지 않고, 참고 지표로만 표와 그래프에 함께 표시합니다.

### 왜 단일 스케일 SSIM이 아니라 MS-SSIM인가

단일 스케일 SSIM은 7~11픽셀 정도의 작은 윈도우 안에서 밝기·대비·구조를 비교합니다. 블록
스크램블처럼 **국소 구조를 직접 파괴**하는 변환에는 잘 반응하지만, 가우시안 블러처럼 넓은
영역이 **국소적으로 평탄**해지는 변환에는 둔감합니다. 윈도우 내 분산이 0에 가까워지면 SSIM
공식의 안정화 상수(C1, C2)가 지배적이 되어, 실제로는 원본과 많이 달라 보이는 이미지도
"거의 동일하다"고 낮게 잡는 경향이 있습니다.

이 노트북에서 실제로 검증한 결과, 이 문제 때문에 블록 단독 최대 점수(~49~50점)와 블러 단독
최대 점수(~25~34점)의 격차가 실제 체감 프라이버시 차이보다 크게 벌어졌습니다.

**MS-SSIM**(Wang, Z., Simoncelli, E. P., & Bovik, A. C. (2003). *Multi-scale structural
similarity for image quality assessment*. Asilomar Conference on Signals, Systems and
Computers)은 이미지를 5단계 해상도로 낮춰가며(매 단계 2×2 평균 풀링) 각 단계의 대비·구조
유사도를 곱해서 하나로 합칩니다. 여러 해상도에서 반복 검증하기 때문에 블러처럼 국소적으로만
평탄해지는 왜곡도 더 일관되게 잡아냅니다. 이 노트북 자체 검증에서, 블러 sigma=256 조건의
DSSIM 기반 점수가 단일 스케일 SSIM 기준 약 34점에서 MS-SSIM 기준 약 44점으로 올라가
블록 계열과의 격차가 크게 줄어드는 것을 확인했습니다.

### 최종 수식

원본 이미지 \(I\), 변환 이미지 \(I'\)에 대해 (그레이스케일 휘도 채널 기준):

\[
MSSSIM_i = MSSSIM(I_i, I'_i)
\]

\[
MS\text{-}DSSIM_i = \frac{1 - MSSSIM_i}{2}
\]

\[
PrivacyScore_i = 100 \times MS\text{-}DSSIM_i
               = 50 \times (1 - MSSSIM_i)
\]

조건별 최종 점수는 이미지별 점수의 평균입니다.

\[
PrivacyScore_{condition} = \frac{1}{N}\sum_{i=1}^{N} PrivacyScore_i
\]

- **이론적 범위**: MS-SSIM도 SSIM과 마찬가지로 \([-1, 1]\) 범위이므로 PrivacyScore는
  이론상 \([0, 100]\)입니다. 다만 자연 이미지는 완전히 무작위로 섞거나(block_size=1) 강하게
  블러를 걸어도 MS-SSIM이 크게 음수로 내려가지는 않으므로, 실제 관측되는 상한은 여전히
  100보다 낮습니다(대략 50점 안팎). 이는 버그가 아니라 DSSIM류 지표의 일반적인 특성입니다.
- 0점에 가까움: 원본과 거의 동일
- `ssim_mean`(단일 스케일 SSIM, 참고), `psnr_mean`/`psnr_std`(참고): 점수 계산에는 사용하지 않음


## 1️⃣ 라이브러리 임포트

In [ ]:
# Cell 1: 라이브러리 임포트
import os, glob, json, math
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from PIL import Image, ImageFilter, ImageOps
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from scipy.ndimage import uniform_filter  # MS-SSIM 로컬 통계 계산용

try:
    from IPython.display import display
except Exception:
    display = print


## 2️⃣ 한글 폰트 설정 (matplotlib 한글 완전 지원 + 다크테마)

In [ ]:
# Cell 2: 한글 폰트 설정 (기존 노트북 Cell 2와 동일)
_CANDIDATES = [
    "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
    "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf",
    "/usr/share/fonts/truetype/unfonts-core/UnDotum.ttf",
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
]
font_path = next((p for p in _CANDIDATES if os.path.exists(p)), None)
if font_path is None:
    for p in fm.findSystemFonts(fontpaths=None, fontext="ttf"):
        if any(kw in p.lower() for kw in ["nanum", "malgun", "gulim", "undotum", "noto"]):
            font_path = p
            break

if font_path is None:
    print("⚠️  한글 폰트를 찾지 못했습니다. 기본 폰트 사용.")
    font_name = "DejaVu Sans"
else:
    cache_dir = matplotlib.get_cachedir()
    for f in glob.glob(os.path.join(cache_dir, "*.json")) + glob.glob(os.path.join(cache_dir, "*.cache")):
        try: os.remove(f)
        except Exception: pass
    fm.fontManager.addfont(font_path)
    font_name = fm.FontProperties(fname=font_path).get_name()
    print(f"✅ 한글 폰트 적용: {font_name}  ({font_path})")

matplotlib.rcParams["font.family"]        = "sans-serif"
matplotlib.rcParams["font.sans-serif"]    = [font_name, "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
matplotlib.rcParams.update({
    "figure.facecolor":  "#0d1117",
    "axes.facecolor":    "#161b22",
    "axes.edgecolor":    "#444",
    "axes.labelcolor":   "#aaa",
    "xtick.color":       "#aaa",
    "ytick.color":       "#aaa",
    "grid.color":        "#333",
    "grid.linestyle":    "--",
    "grid.alpha":        0.5,
    "legend.facecolor":  "#222",
    "legend.labelcolor": "white",
    "text.color":        "white",
})
print("✅ 다크테마 rcParams 전역 적용 완료")

## 3️⃣ 실험 설정

In [ ]:
# Cell 3: 실험 설정 - 경로는 기존 노트북 값을 그대로 사용
BASE_TRAIN   = "/workspace/Train"      # 원본 이미지 루트. Docker/VSCode 컨테이너 기준 경로
IMG_SIZE     = 512
SEED         = 42
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp")

BLOCK_SIZES  = [512, 256, 128, 64, 32, 16, 8, 4, 2, 1]
BLUR_SIGMAS  = [0, 1, 2, 4, 8, 16, 32, 64, 128, 256]

# 속도 조절용. None이면 전체 이미지 사용, 예: 20이면 클래스별 최대 20장만 사용
N_SAMPLES_PER_CLASS = None

# 기존 출력 경로 유지 (CSV/JSON/그래프/표)
OUT_DIR   = "./privacy_score_outputs"
GRAPH_DIR = os.path.join(OUT_DIR, "eval_graphs_privacy_score")
TABLE_DIR = os.path.join(OUT_DIR, "tables")

for _dir in (OUT_DIR, GRAPH_DIR, TABLE_DIR):
    os.makedirs(_dir, exist_ok=True)

# SSIM 계산에 사용한 샘플 사진은 조건별 폴더로 나눠 바탕화면에 저장
# 예: 바탕화면/privacy_sample_images/블록/512/, .../블러/16/, .../조합/512/16/
DESKTOP_DIR    = os.path.expanduser("~/Desktop")
SAMPLE_IMG_DIR = os.path.join(DESKTOP_DIR, "privacy_sample_images")
os.makedirs(SAMPLE_IMG_DIR, exist_ok=True)

print(f"BASE_TRAIN: {BASE_TRAIN}")
print(f"OUT_DIR   : {OUT_DIR}")
print(f"GRAPH_DIR : {GRAPH_DIR}")
print(f"TABLE_DIR : {TABLE_DIR}")
print(f"SAMPLE_IMG_DIR (바탕화면): {SAMPLE_IMG_DIR}")
print(f"실험 조건: 블록 {len(BLOCK_SIZES)}개, 블러 {len(BLUR_SIGMAS)}개, 조합 {len(BLOCK_SIZES) * len(BLUR_SIGMAS)}개")


## 3️⃣-1 조건별 샘플 이미지 저장 헬퍼 (바탕화면)


In [ ]:
# Cell 3b: SSIM 계산에 사용한 사진을 조건별 폴더에 저장하는 헬퍼
# 폴더 구조 예: 바탕화면/privacy_sample_images/블록/512/, .../블러/16/, .../조합/512/16/
# 폴더 하나당 원본, 변환본, 지표(txt) 총 3개 파일을 저장합니다.
def save_condition_sample(type_name, param_parts, original_img, transformed_img, metrics):
    """
    type_name   : "블록" / "블러" / "조합"
    param_parts : 폴더 depth를 구성하는 파라미터 값 리스트. 예: [512] 또는 [512, 16]
    original_img, transformed_img : np.ndarray (H, W, 3) uint8
    metrics     : {"ssim_mean": ..., "psnr_mean_ref": ..., "privacy_score": ...} 형태의 dict
    """
    folder = os.path.join(SAMPLE_IMG_DIR, type_name, *[str(p) for p in param_parts])
    os.makedirs(folder, exist_ok=True)

    Image.fromarray(original_img).save(os.path.join(folder, "01_original.png"))
    Image.fromarray(transformed_img).save(os.path.join(folder, "02_transformed.png"))

    with open(os.path.join(folder, "03_metrics.txt"), "w", encoding="utf-8") as f:
        for k, v in metrics.items():
            f.write(f"{k}: {v}\n")

    return folder

print("✅ save_condition_sample() 정의 완료 — 조건마다 원본/변환/지표 3개 파일을 폴더별로 저장합니다.")


## 4️⃣ 변환 함수 (기존 정확도 실험 노트북과 완전히 동일)

In [ ]:
# Cell 4: block_scramble / apply_gaussian_blur — 재현성을 위해 그대로 재사용
def block_scramble(img_array, seed, block_size):
    h, w, c = img_array.shape
    if block_size >= min(h, w):
        return img_array.copy()

    pad_h = (block_size - h % block_size) % block_size
    pad_w = (block_size - w % block_size) % block_size
    if pad_h > 0 or pad_w > 0:
        img_array = np.pad(img_array, ((0, pad_h), (0, pad_w), (0, 0)), mode="reflect")

    ph, pw   = img_array.shape[:2]
    n_h, n_w = ph // block_size, pw // block_size
    blocks   = [img_array[i*block_size:(i+1)*block_size, j*block_size:(j+1)*block_size]
                for i in range(n_h) for j in range(n_w)]

    rng    = np.random.default_rng(seed)
    order  = rng.permutation(n_h * n_w)
    result = np.zeros_like(img_array)

    for new_idx, old_idx in enumerate(order):
        i, j = new_idx // n_w, new_idx % n_w
        result[i*block_size:(i+1)*block_size, j*block_size:(j+1)*block_size] = blocks[old_idx]

    return result[:h, :w]


def apply_gaussian_blur(img_array, sigma):
    if sigma == 0:
        return img_array.copy()
    pil_img = Image.fromarray(img_array.astype(np.uint8))
    blurred = pil_img.filter(ImageFilter.GaussianBlur(radius=float(sigma)))
    return np.array(blurred, dtype=np.uint8)

print("✅ block_scramble / apply_gaussian_blur 정의 완료")

## 5️⃣ 원본 이미지 로딩

In [ ]:
# Cell 5: 원본 이미지 로딩
def _is_image_file(path):
    return str(path).lower().endswith(IMAGE_EXTENSIONS)

def _is_ignored_path(path):
    parts = set(os.path.normpath(path).split(os.sep))
    ignored = {"privacy_score_outputs", "eval_graphs_privacy_score", "sample_images", ".ipynb_checkpoints", "__pycache__"}
    return bool(parts & ignored)

def collect_image_files(base_dir):
    records = []
    for root, _, files in os.walk(base_dir):
        if _is_ignored_path(root):
            continue
        for f in sorted(files):
            if not _is_image_file(f):
                continue
            path = os.path.join(root, f)
            rel = os.path.relpath(path, base_dir)
            first_part = rel.split(os.sep)[0]
            label = first_part if first_part != os.path.basename(rel) else "root"
            records.append({"path": path, "relative_path": rel, "label": label})
    return pd.DataFrame(records)

def load_original_images(base_dir, n_per_class=None, seed=SEED):
    assert os.path.exists(base_dir), f"BASE_TRAIN 경로가 없습니다: {base_dir}"

    image_table = collect_image_files(base_dir)
    assert len(image_table) > 0, f"이미지 파일을 찾을 수 없습니다: {base_dir}"

    print(f"이미지 파일 발견: {len(image_table)}장")
    class_counts = image_table.groupby("label").size().rename("n_images").reset_index()
    display(class_counts)

    if n_per_class is not None:
        image_table = (
            image_table.groupby("label", group_keys=False)
            .apply(lambda x: x.sample(n=min(len(x), n_per_class), random_state=seed))
            .reset_index(drop=True)
        )
        print(f"샘플링 후 이미지: {len(image_table)}장")

    image_table = image_table.sort_values(["label", "relative_path"]).reset_index(drop=True)
    image_table.to_csv(os.path.join(TABLE_DIR, "selected_images.csv"), index=False, encoding="utf-8-sig")

    images = []
    failed = []
    for _, row in image_table.iterrows():
        try:
            img = Image.open(row["path"])
            img = ImageOps.exif_transpose(img).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
            images.append(np.array(img, dtype=np.uint8))
        except Exception as e:
            failed.append((row["path"], str(e)))

    if failed:
        print(f"로드 실패 {len(failed)}장")
        for path, err in failed[:10]:
            print(f"  - {path}: {err}")

    print(f"원본 이미지 {len(images)}장 로드 완료 (경로: {base_dir})")
    return images, image_table

images, image_table = load_original_images(BASE_TRAIN, n_per_class=N_SAMPLES_PER_CLASS, seed=SEED)
assert len(images) > 0, f"원본 이미지를 찾을 수 없습니다: {BASE_TRAIN}"

Image.fromarray(images[0]).save(os.path.join(SAMPLE_IMG_DIR, "00_original_sample.png"))
print(f"원본 샘플 이미지 저장: {os.path.join(SAMPLE_IMG_DIR, '00_original_sample.png')}")


## 6️⃣ SSIM/PSNR + Privacy Score 계산 함수

In [ ]:
# Cell 6: MS-SSIM 기반 Privacy Score + 단일 스케일 SSIM/PSNR 참고 지표 계산
# Wang, Simoncelli & Bovik (2003), "Multi-scale structural similarity for image
# quality assessment" 공식을 그대로 구현. 그레이스케일 휘도 채널 기준, 5단계
# 해상도(2x2 평균 풀링)에서 각 단계의 contrast*structure를 곱하고, 마지막(가장 저해상도)
# 단계에서만 luminance를 곱해 최종 MS-SSIM을 얻는다 (원 논문의 표준 결합 방식).
_MS_SSIM_WEIGHTS = np.array([0.0448, 0.2856, 0.3001, 0.2363, 0.1333])  # 원 논문 5단계 가중치


def _to_gray(img):
    img = img.astype(np.float64)
    return 0.299 * img[..., 0] + 0.587 * img[..., 1] + 0.114 * img[..., 2]


def _downsample_avg2x2(img):
    h, w = img.shape
    h2, w2 = h - h % 2, w - w % 2
    img = img[:h2, :w2]
    return img.reshape(h2 // 2, 2, w2 // 2, 2).mean(axis=(1, 3))


def _ssim_l_cs(img1, img2, win_size=11, data_range=255.0):
    """단일 스케일에서 SSIM의 luminance(l) 성분과 contrast*structure(cs) 성분을 분리 계산."""
    win_size = min(win_size, img1.shape[0], img1.shape[1])
    if win_size % 2 == 0:
        win_size -= 1
    win_size = max(win_size, 3)

    C1 = (0.01 * data_range) ** 2
    C2 = (0.03 * data_range) ** 2
    mu1 = uniform_filter(img1, size=win_size)
    mu2 = uniform_filter(img2, size=win_size)
    mu1_sq, mu2_sq, mu1_mu2 = mu1 * mu1, mu2 * mu2, mu1 * mu2
    sigma1_sq = uniform_filter(img1 * img1, size=win_size) - mu1_sq
    sigma2_sq = uniform_filter(img2 * img2, size=win_size) - mu2_sq
    sigma12   = uniform_filter(img1 * img2, size=win_size) - mu1_mu2

    l_map = (2 * mu1_mu2 + C1) / (mu1_sq + mu2_sq + C1)
    cs_map = (2 * sigma12 + C2) / (sigma1_sq + sigma2_sq + C2)
    return float(np.mean(l_map)), float(np.mean(cs_map))


def compute_ms_ssim(orig_rgb, transformed_rgb, weights=_MS_SSIM_WEIGHTS, win_size=11, data_range=255.0):
    """MS-SSIM (Wang, Simoncelli & Bovik, 2003). RGB -> 그레이스케일 휘도 기준으로 계산."""
    g1 = _to_gray(orig_rgb)
    g2 = _to_gray(transformed_rgb)

    cs_values = []
    l_final = 1.0
    n_levels = len(weights)
    for level in range(n_levels):
        l_val, cs_val = _ssim_l_cs(g1, g2, win_size=win_size, data_range=data_range)
        cs_values.append(max(cs_val, 1e-8))
        l_final = l_val
        if level < n_levels - 1:
            g1 = _downsample_avg2x2(g1)
            g2 = _downsample_avg2x2(g2)

    cs_values = np.array(cs_values)
    ms_ssim_val = (
        (max(l_final, 1e-8) ** weights[-1])
        * np.prod(cs_values[:-1] ** weights[:-1])
        * (cs_values[-1] ** weights[-1])
    )
    return float(np.clip(ms_ssim_val, -1.0, 1.0))


def compute_ssim_psnr_msssim(orig, transformed):
    """단일 스케일 SSIM(참고), PSNR(참고), MS-SSIM(점수 산정 기준)을 함께 계산."""
    s = ssim_fn(orig, transformed, channel_axis=-1, data_range=255)
    try:
        p = psnr_fn(orig, transformed, data_range=255)
        if np.isinf(p):
            p = 100.0
    except Exception:
        p = 100.0
    mss = compute_ms_ssim(orig, transformed)
    return float(s), float(p), float(mss)


def privacy_score_from_ms_ssim(ms_ssim_value):
    ms_dssim = np.clip((1.0 - float(ms_ssim_value)) / 2.0, 0.0, 1.0)
    return float(ms_dssim * 100.0)


def evaluate_condition(images, transform_fn, **kwargs):
    ssims, psnrs, ms_ssims, scores = [], [], [], []
    for img in images:
        t = transform_fn(img, **kwargs)
        s, p, mss = compute_ssim_psnr_msssim(img, t)
        ssims.append(s)
        psnrs.append(p)
        ms_ssims.append(mss)
        scores.append(privacy_score_from_ms_ssim(mss))

    return {
        "ssim_mean": float(np.mean(ssims)),
        "ssim_std": float(np.std(ssims)),
        "ms_ssim_mean": float(np.mean(ms_ssims)),
        "ms_ssim_std": float(np.std(ms_ssims)),
        "psnr_mean": float(np.mean(psnrs)),
        "psnr_std": float(np.std(psnrs)),
        "dssim_mean": float(np.mean(scores) / 100.0),
        "privacy_score": float(np.mean(scores)),
        "privacy_score_std": float(np.std(scores)),
    }


print("MS-SSIM 기반 Privacy Score 함수 정의 완료 (Wang, Simoncelli & Bovik, 2003)")
print("Privacy Score = 100 * ((1 - MS-SSIM) / 2)")
print("단일 스케일 ssim_mean / psnr_mean은 참고 지표로만 저장 (점수 계산에는 미사용)")


## 7️⃣ 🧩 블록 암호화 단독 실행 (10개)

In [ ]:
# Cell 7: 블록 암호화 단독
rows = []
print("블록 암호화 단독 SSIM/PSNR 계산 중...")
for bs in BLOCK_SIZES:
    m = evaluate_condition(images, block_scramble, seed=SEED, block_size=bs)
    rows.append({"condition": f"B{bs}_S0", "type": "block", "block_size": bs, "sigma": 0, **m})
    print(
        f"  Block{bs:>4}: Score={m['privacy_score']:6.2f}/100  "
        f"SSIM={m['ssim_mean']:.4f}  PSNR(ref)={m['psnr_mean']:6.2f}dB"
    )

    sample = block_scramble(images[0], SEED, bs)
    save_condition_sample(
        "블록", [bs], images[0], sample,
        {"block_size": bs, "ssim_mean": m["ssim_mean"], "psnr_mean_ref": m["psnr_mean"],
         "privacy_score": m["privacy_score"]},
    )

print(f"블록 샘플 폴더 {len(BLOCK_SIZES)}개 저장: {os.path.join(SAMPLE_IMG_DIR, '블록')}")


## 8️⃣ 🌫️ 가우시안 블러 단독 실행 (10개)

In [ ]:
# Cell 8: 가우시안 블러 단독
print("가우시안 블러 단독 SSIM/PSNR 계산 중...")
for sg in BLUR_SIGMAS:
    m = evaluate_condition(images, apply_gaussian_blur, sigma=sg)
    rows.append({"condition": f"B512_S{sg}", "type": "blur", "block_size": 512, "sigma": sg, **m})
    print(
        f"  Sigma{sg:>4}: Score={m['privacy_score']:6.2f}/100  "
        f"SSIM={m['ssim_mean']:.4f}  PSNR(ref)={m['psnr_mean']:6.2f}dB"
    )

    sample = apply_gaussian_blur(images[0], sg)
    save_condition_sample(
        "블러", [sg], images[0], sample,
        {"sigma": sg, "ssim_mean": m["ssim_mean"], "psnr_mean_ref": m["psnr_mean"],
         "privacy_score": m["privacy_score"]},
    )

print(f"블러 샘플 폴더 {len(BLUR_SIGMAS)}개 저장: {os.path.join(SAMPLE_IMG_DIR, '블러')}")


## 9️⃣ 🧩🌫️ 블록+블러 조합 실행 (100개)

In [ ]:
# Cell 9: 블록+블러 조합 (10 x 10 = 100개)
print("블록+블러 조합 SSIM/PSNR 계산 중...")
for bs in BLOCK_SIZES:
    for sg in BLUR_SIGMAS:
        def combo_fn(img, block_size=bs, sigma=sg):
            return apply_gaussian_blur(block_scramble(img, SEED, block_size), sigma)

        m = evaluate_condition(images, combo_fn)
        rows.append({"condition": f"B{bs}_S{sg}", "type": "combo", "block_size": bs, "sigma": sg, **m})
        print(
            f"  B{bs:>4}_S{sg:<4}: Score={m['privacy_score']:6.2f}/100  "
            f"SSIM={m['ssim_mean']:.4f}  PSNR(ref)={m['psnr_mean']:6.2f}dB"
        )

        sample = combo_fn(images[0])
        save_condition_sample(
            "조합", [bs, sg], images[0], sample,
            {"block_size": bs, "sigma": sg, "ssim_mean": m["ssim_mean"], "psnr_mean_ref": m["psnr_mean"],
             "privacy_score": m["privacy_score"]},
        )

print(f"조합 샘플 폴더 {len(BLOCK_SIZES) * len(BLUR_SIGMAS)}개 저장: {os.path.join(SAMPLE_IMG_DIR, '조합')}")
print(f"전체 {len(rows)}개 조건 계산 완료")


## 🔟 결과 저장 (CSV / JSON)

In [ ]:
# Cell 10: 결과 DataFrame 생성 및 저장
df = pd.DataFrame(rows)

preferred_cols = [
    "condition", "type", "block_size", "sigma", "privacy_score", "privacy_score_std",
    "dssim_mean", "ms_ssim_mean", "ms_ssim_std", "ssim_mean", "ssim_std", "psnr_mean", "psnr_std"
]
df = df[[c for c in preferred_cols if c in df.columns] + [c for c in df.columns if c not in preferred_cols]]
df_sorted = df.sort_values("privacy_score", ascending=False).reset_index(drop=True)

summary_by_type = (
    df.groupby("type")
      .agg(
          n_conditions=("condition", "count"),
          score_mean=("privacy_score", "mean"),
          score_max=("privacy_score", "max"),
          score_min=("privacy_score", "min"),
          ms_ssim_mean=("ms_ssim_mean", "mean"),
          ssim_mean=("ssim_mean", "mean"),
          psnr_mean_ref=("psnr_mean", "mean"),
      )
      .reset_index()
      .sort_values("score_mean", ascending=False)
)

best_by_type = (
    df.loc[df.groupby("type")["privacy_score"].idxmax()]
      .sort_values("privacy_score", ascending=False)
      .reset_index(drop=True)
)

df_sorted.to_csv(os.path.join(OUT_DIR, "privacy_scores.csv"), index=False, encoding="utf-8-sig")
df_sorted.to_json(os.path.join(OUT_DIR, "privacy_scores.json"), orient="records", force_ascii=False, indent=2)
summary_by_type.to_csv(os.path.join(TABLE_DIR, "summary_by_type.csv"), index=False, encoding="utf-8-sig")
best_by_type.to_csv(os.path.join(TABLE_DIR, "best_by_type.csv"), index=False, encoding="utf-8-sig")

try:
    with pd.ExcelWriter(os.path.join(OUT_DIR, "privacy_scores.xlsx")) as writer:
        df_sorted.to_excel(writer, sheet_name="all_scores", index=False)
        summary_by_type.to_excel(writer, sheet_name="summary_by_type", index=False)
        best_by_type.to_excel(writer, sheet_name="best_by_type", index=False)
    print(f"Excel 저장 완료: {os.path.join(OUT_DIR, 'privacy_scores.xlsx')}")
except Exception as e:
    print(f"Excel 저장은 건너뜀: {e}")

print(f"프라이버시 점수 저장 완료: {os.path.join(OUT_DIR, 'privacy_scores.csv')}")
print(f"요약표 저장 완료: {TABLE_DIR}")
display(summary_by_type)
display(df_sorted.head(15))


## 11. 상세 그래프 생성

아래 그래프들은 `GRAPH_DIR`에 PNG로 저장됩니다. Privacy Score는 SSIM 기반 점수이며, PSNR은 참고 지표로만 함께 표시합니다.


In [ ]:
# Cell 11: 상세 그래프 공통 설정
ORANGE, CYAN, GREEN, RED, GRAY, PURPLE, BLUE = "#f08c00", "#1098ad", "#2f9e44", "#e03131", "#868e96", "#7048e8", "#1971c2"

def _save(fig, name):
    path = os.path.join(GRAPH_DIR, name)
    fig.tight_layout()
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()
    print(f"저장: {path}")

def _annotate_bars(ax, fmt="{:.1f}", dy=0.5, fontsize=8):
    for patch in ax.patches:
        h = patch.get_height()
        if np.isnan(h):
            continue
        ax.text(
            patch.get_x() + patch.get_width() / 2,
            h + dy,
            fmt.format(h),
            ha="center",
            va="bottom",
            fontsize=fontsize,
            rotation=0,
        )

block_df = df[df["type"] == "block"].sort_values("block_size", ascending=False).copy()
blur_df  = df[df["type"] == "blur"].sort_values("sigma").copy()
combo_df = df[df["type"] == "combo"].copy()

pivot_score   = combo_df.pivot(index="block_size", columns="sigma", values="privacy_score").reindex(index=BLOCK_SIZES[::-1], columns=BLUR_SIGMAS)
pivot_msssim  = combo_df.pivot(index="block_size", columns="sigma", values="ms_ssim_mean").reindex(index=BLOCK_SIZES[::-1], columns=BLUR_SIGMAS)
pivot_ssim    = combo_df.pivot(index="block_size", columns="sigma", values="ssim_mean").reindex(index=BLOCK_SIZES[::-1], columns=BLUR_SIGMAS)
pivot_psnr    = combo_df.pivot(index="block_size", columns="sigma", values="psnr_mean").reindex(index=BLOCK_SIZES[::-1], columns=BLUR_SIGMAS)


In [ ]:
# 01. 블록 픽셀 크기별 Privacy Score + SSIM + PSNR 참고 (Original을 맨 왼쪽에 표시)
block_labels = ["Original"] + block_df["block_size"].astype(int).astype(str).tolist()
block_scores = [0.0] + block_df["privacy_score"].tolist()
block_ssims  = [1.0] + block_df["ssim_mean"].tolist()
block_psnrs  = [100.0] + block_df["psnr_mean"].tolist()  # Original은 원본=원본이므로 참고용 상한값(100) 사용

fig, ax1 = plt.subplots(figsize=(10.6, 5.5))
x = np.arange(len(block_labels))
bars = ax1.bar(x, block_scores, color=GREEN, alpha=0.82, label="Privacy Score")
ax1.set_xticks(x)
ax1.set_xticklabels(block_labels)
ax1.set_xlabel("Block size (pixels) — Original 기준 좌측부터")
ax1.set_ylabel("Privacy Score (0-100, SSIM only)", color=GREEN)
ax1.set_ylim(0, max(5, max(block_scores) * 1.18))
ax1.grid(axis="y")
_annotate_bars(ax1, "{:.1f}", dy=max(0.25, max(block_scores) * 0.015))

ax2 = ax1.twinx()
ax2.plot(x, block_ssims, "o-", color=CYAN, linewidth=2, label="SSIM")
ax2.plot(x, np.array(block_psnrs) / 100.0, "s--", color=ORANGE, linewidth=1.8, label="PSNR/100 ref")
ax2.set_ylabel("SSIM / PSNR divided by 100", color=CYAN)
ax2.set_ylim(0, max(1.05, max(block_ssims) * 1.1))

lines1, names1 = ax1.get_legend_handles_labels()
lines2, names2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, names1 + names2, loc="upper left")
ax1.set_title("01. Block Size Detail: Privacy Score from SSIM, PSNR as Reference")
_save(fig, "01_block_detail_score_ssim_psnr_ref.png")


In [ ]:
# 02. 블러 sigma별 Privacy Score + SSIM + PSNR 참고 (Original을 맨 왼쪽에 표시)
blur_labels = ["Original"] + blur_df["sigma"].astype(int).astype(str).tolist()
blur_scores = [0.0] + blur_df["privacy_score"].tolist()
blur_ssims  = [1.0] + blur_df["ssim_mean"].tolist()
blur_psnrs  = [100.0] + blur_df["psnr_mean"].tolist()  # Original은 원본=원본이므로 참고용 상한값(100) 사용

fig, ax1 = plt.subplots(figsize=(10.6, 5.5))
x = np.arange(len(blur_labels))
bars = ax1.bar(x, blur_scores, color=GREEN, alpha=0.82, label="Privacy Score")
ax1.set_xticks(x)
ax1.set_xticklabels(blur_labels)
ax1.set_xlabel("Gaussian blur sigma — Original 기준 좌측부터")
ax1.set_ylabel("Privacy Score (0-100, SSIM only)", color=GREEN)
ax1.set_ylim(0, max(5, max(blur_scores) * 1.18))
ax1.grid(axis="y")
_annotate_bars(ax1, "{:.1f}", dy=max(0.25, max(blur_scores) * 0.015))

ax2 = ax1.twinx()
ax2.plot(x, blur_ssims, "o-", color=CYAN, linewidth=2, label="SSIM")
ax2.plot(x, np.array(blur_psnrs) / 100.0, "s--", color=ORANGE, linewidth=1.8, label="PSNR/100 ref")
ax2.set_ylabel("SSIM / PSNR divided by 100", color=CYAN)
ax2.set_ylim(0, max(1.05, max(blur_ssims) * 1.1))

lines1, names1 = ax1.get_legend_handles_labels()
lines2, names2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, names1 + names2, loc="upper left")
ax1.set_title("02. Blur Sigma Detail: Privacy Score from SSIM, PSNR as Reference")
_save(fig, "02_blur_detail_score_ssim_psnr_ref.png")


In [ ]:
# 03. 블록+블러 조합 Privacy Score 히트맵 - 셀마다 점수 표시
fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(pivot_score.values, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(BLUR_SIGMAS)))
ax.set_xticklabels(BLUR_SIGMAS)
ax.set_yticks(range(len(BLOCK_SIZES)))
ax.set_yticklabels(BLOCK_SIZES[::-1])
ax.set_xlabel("Gaussian blur sigma")
ax.set_ylabel("Block size (pixels)")
ax.set_title("03. Block + Blur Combination Privacy Score Heatmap")
for i in range(pivot_score.shape[0]):
    for j in range(pivot_score.shape[1]):
        value = pivot_score.values[i, j]
        color = "white" if value > np.nanmax(pivot_score.values) * 0.45 else "black"
        ax.text(j, i, f"{value:.1f}", ha="center", va="center", fontsize=7, color=color)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Privacy Score (0-100, SSIM only)")
_save(fig, "03_combo_privacy_score_heatmap_annotated.png")


In [ ]:
# 04. 조합 조건 단일 스케일 SSIM 히트맵 - 참고 지표 (점수는 MS-SSIM 기준, 04b 참고)
fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(pivot_ssim.values, cmap="magma_r", aspect="auto")
ax.set_xticks(range(len(BLUR_SIGMAS)))
ax.set_xticklabels(BLUR_SIGMAS)
ax.set_yticks(range(len(BLOCK_SIZES)))
ax.set_yticklabels(BLOCK_SIZES[::-1])
ax.set_xlabel("Gaussian blur sigma")
ax.set_ylabel("Block size (pixels)")
ax.set_title("04. Block + Blur SSIM Mean Heatmap (참고, 점수 산정에는 미사용)")
for i in range(pivot_ssim.shape[0]):
    for j in range(pivot_ssim.shape[1]):
        value = pivot_ssim.values[i, j]
        ax.text(j, i, f"{value:.3f}", ha="center", va="center", fontsize=6.5, color="white" if value < 0.45 else "black")
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("SSIM mean")
_save(fig, "04_combo_ssim_heatmap_annotated.png")


In [ ]:
# 04b. 조합 조건 MS-SSIM 히트맵 - privacy_score의 실제 원천 지표
fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(pivot_msssim.values, cmap="magma_r", aspect="auto")
ax.set_xticks(range(len(BLUR_SIGMAS)))
ax.set_xticklabels(BLUR_SIGMAS)
ax.set_yticks(range(len(BLOCK_SIZES)))
ax.set_yticklabels(BLOCK_SIZES[::-1])
ax.set_xlabel("Gaussian blur sigma")
ax.set_ylabel("Block size (pixels)")
ax.set_title("04b. Block + Blur MS-SSIM Mean Heatmap (privacy_score의 원천 지표)")
for i2 in range(pivot_msssim.shape[0]):
    for j2 in range(pivot_msssim.shape[1]):
        value = pivot_msssim.values[i2, j2]
        ax.text(j2, i2, f"{value:.3f}", ha="center", va="center", fontsize=6.5, color="white" if value < 0.45 else "black")
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("MS-SSIM mean")
_save(fig, "04b_combo_msssim_heatmap_annotated.png")


In [ ]:
# 05. 조합 조건 PSNR 참고 히트맵 - 점수에는 미사용
fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(pivot_psnr.values, cmap="cividis", aspect="auto")
ax.set_xticks(range(len(BLUR_SIGMAS)))
ax.set_xticklabels(BLUR_SIGMAS)
ax.set_yticks(range(len(BLOCK_SIZES)))
ax.set_yticklabels(BLOCK_SIZES[::-1])
ax.set_xlabel("Gaussian blur sigma")
ax.set_ylabel("Block size (pixels)")
ax.set_title("05. Block + Blur PSNR Reference Heatmap (not used in score)")
for i in range(pivot_psnr.shape[0]):
    for j in range(pivot_psnr.shape[1]):
        value = pivot_psnr.values[i, j]
        ax.text(j, i, f"{value:.1f}", ha="center", va="center", fontsize=6.5, color="white" if value < np.nanmedian(pivot_psnr.values) else "black")
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("PSNR mean (dB, reference only)")
_save(fig, "05_combo_psnr_reference_heatmap_annotated.png")


In [ ]:
# 06. 전체 조건 Privacy Score 상위 랭킹
top_n = min(20, len(df_sorted))
top_df = df_sorted.head(top_n).iloc[::-1].copy()
method_colors = {"block": PURPLE, "blur": ORANGE, "combo": GREEN}
fig, ax = plt.subplots(figsize=(11, max(6, top_n * 0.36)))
ax.barh(top_df["condition"], top_df["privacy_score"], color=[method_colors.get(t, GRAY) for t in top_df["type"]])
ax.set_xlabel("Privacy Score (0-100, SSIM only)")
ax.set_ylabel("Condition")
ax.set_title(f"06. Top {top_n} Privacy Conditions")
ax.grid(axis="x")
for y, value in enumerate(top_df["privacy_score"]):
    ax.text(value + max(0.2, df_sorted["privacy_score"].max() * 0.01), y, f"{value:.2f}", va="center", fontsize=8)
_save(fig, "06_top_conditions_privacy_score.png")


In [ ]:
# 07. MS-SSIM과 Privacy Score 관계 - 이제 MS-SSIM이 점수를 결정함
fig, ax = plt.subplots(figsize=(8, 6))
for t, color in method_colors.items():
    sub = df[df["type"] == t]
    ax.scatter(sub["ms_ssim_mean"], sub["privacy_score"], label=t, color=color, s=60, alpha=0.78, edgecolors="white")
ax.set_xlabel("MS-SSIM mean")
ax.set_ylabel("Privacy Score = 50 x (1 - MS-SSIM)")
ax.set_title("07. MS-SSIM Determines Privacy Score")
ax.grid(True)
ax.legend(title="type")
for _, r in df_sorted.head(8).iterrows():
    ax.annotate(r["condition"], (r["ms_ssim_mean"], r["privacy_score"]), fontsize=7, xytext=(4, 4), textcoords="offset points")
_save(fig, "07_msssim_vs_privacy_score.png")


In [ ]:
# 07b. (비교) 단일 스케일 SSIM vs Privacy Score - 왜 MS-SSIM으로 바꿨는지 시각적 근거
fig, ax = plt.subplots(figsize=(8, 6))
for t, color in method_colors.items():
    sub = df[df["type"] == t]
    ax.scatter(sub["ssim_mean"], sub["privacy_score"], label=t, color=color, s=60, alpha=0.78, edgecolors="white")
ax.set_xlabel("SSIM mean (single-scale, 참고)")
ax.set_ylabel("Privacy Score (MS-SSIM 기준)")
ax.set_title("07b. Single-scale SSIM vs MS-SSIM-based Privacy Score (참고용 비교)")
ax.grid(True)
ax.legend(title="type")
_save(fig, "07b_singlescale_ssim_vs_privacy_score_reference.png")


In [ ]:
# 08. PSNR 참고 지표와 Privacy Score 관계 - 점수 계산에는 미사용
fig, ax = plt.subplots(figsize=(8, 6))
for t, color in method_colors.items():
    sub = df[df["type"] == t]
    ax.scatter(sub["psnr_mean"], sub["privacy_score"], label=t, color=color, s=60, alpha=0.78, edgecolors="white")
ax.axvline(40, color=RED, linestyle="--", linewidth=1, label="40 dB reference")
ax.axvline(15, color=BLUE, linestyle="--", linewidth=1, label="15 dB reference")
ax.set_xlabel("PSNR mean (dB, reference only)")
ax.set_ylabel("Privacy Score (0-100, SSIM only)")
ax.set_title("08. PSNR Reference vs Privacy Score")
ax.grid(True)
ax.legend(fontsize=8)
_save(fig, "08_psnr_reference_vs_privacy_score.png")


In [ ]:
# 09. 변환 유형별 Privacy Score 분포
fig, ax = plt.subplots(figsize=(8, 5.5))
data = [df[df["type"] == t]["privacy_score"].values for t in ["block", "blur", "combo"]]
bp = ax.boxplot(data, labels=["block", "blur", "combo"], patch_artist=True, showmeans=True)
for patch, color in zip(bp["boxes"], [PURPLE, ORANGE, GREEN]):
    patch.set_facecolor(color)
    patch.set_alpha(0.55)
ax.set_ylabel("Privacy Score (0-100, SSIM only)")
ax.set_title("09. Privacy Score Distribution by Transform Type")
ax.grid(axis="y")
_save(fig, "09_type_distribution_boxplot.png")


In [ ]:
# 10. 상위 조건 표를 이미지로 저장
table_df = df_sorted.head(15)[["condition", "type", "block_size", "sigma", "privacy_score", "ssim_mean", "psnr_mean"]].copy()
for col, digits in [("privacy_score", 2), ("ssim_mean", 4), ("psnr_mean", 2)]:
    table_df[col] = table_df[col].map(lambda x: f"{x:.{digits}f}")
for col in ["block_size", "sigma"]:
    table_df[col] = table_df[col].map(lambda x: "" if pd.isna(x) else f"{float(x):.0f}")

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.axis("off")
ax.set_title("10. Top 15 Conditions Table", fontsize=14, pad=14)
tbl = ax.table(cellText=table_df.values, colLabels=table_df.columns, loc="center", cellLoc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(8.5)
tbl.scale(1, 1.25)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor("#dee2e6")
        cell.set_text_props(weight="bold")
    else:
        cell.set_facecolor("#ffffff" if r % 2 else "#f8f9fa")
_save(fig, "10_top15_conditions_table.png")


In [ ]:
# 11. 원본 vs 각 유형 최고 조건 샘플 비교
def _combo_transform(img, block_size, sigma):
    return apply_gaussian_blur(block_scramble(img, SEED, int(block_size)), float(sigma))

sample_original = images[0]
best_rows = best_by_type.sort_values("type").reset_index(drop=True)

fig, axes = plt.subplots(1, len(best_rows) + 1, figsize=(4.2 * (len(best_rows) + 1), 4.5))
axes[0].imshow(sample_original)
axes[0].set_title("Original")
axes[0].axis("off")

for ax, (_, r) in zip(axes[1:], best_rows.iterrows()):
    if r["type"] == "block":
        transformed = block_scramble(sample_original, SEED, int(r["block_size"]))
    elif r["type"] == "blur":
        transformed = apply_gaussian_blur(sample_original, float(r["sigma"]))
    else:
        transformed = _combo_transform(sample_original, r["block_size"], r["sigma"])
    ax.imshow(transformed)
    ax.set_title(f"{r['type']} best\n{r['condition']}\nScore {r['privacy_score']:.2f}")
    ax.axis("off")

fig.suptitle("11. Sample Comparison: Original vs Best Conditions", fontsize=14)
_save(fig, "11_sample_original_vs_best_conditions.png")


In [ ]:
# 12. 종합 대시보드
fig = plt.figure(figsize=(15, 10))
gs = GridSpec(2, 3, figure=fig)

ax = fig.add_subplot(gs[0, 0])
# block_df가 이미 block_size 내림차순(512 -> 1)이라 Original에 가까운 조건이 자연스럽게 왼쪽에 옴.
# 기존 invert_xaxis()는 순서를 반대로 뒤집어 원본과 먼 조건(1)이 왼쪽에 오는 오류가 있어 제거함.
ax.plot(block_df["block_size"].astype(str), block_df["privacy_score"], "o-", color=GREEN)
ax.set_title("Block score (Original-like -> left)")
ax.set_xlabel("block size")
ax.set_ylabel("score")
ax.grid(True)

ax = fig.add_subplot(gs[0, 1])
ax.plot(blur_df["sigma"].astype(str), blur_df["privacy_score"], "o-", color=GREEN)
ax.set_title("Blur score")
ax.set_xlabel("sigma")
ax.set_ylabel("score")
ax.grid(True)

ax = fig.add_subplot(gs[0, 2])
im = ax.imshow(pivot_score.values, cmap="viridis", aspect="auto")
ax.set_title("Combo score heatmap")
ax.set_xticks(range(len(BLUR_SIGMAS)))
ax.set_xticklabels(BLUR_SIGMAS, fontsize=7)
ax.set_yticks(range(len(BLOCK_SIZES)))
ax.set_yticklabels(BLOCK_SIZES[::-1], fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.046)

ax = fig.add_subplot(gs[1, 0])
for t, color in method_colors.items():
    sub = df[df["type"] == t]
    ax.scatter(sub["ms_ssim_mean"], sub["privacy_score"], label=t, color=color, s=35, alpha=0.75)
ax.set_title("MS-SSIM vs score (실제 결정 관계)")
ax.set_xlabel("MS-SSIM")
ax.set_ylabel("score")
ax.grid(True)
ax.legend(fontsize=8)

ax = fig.add_subplot(gs[1, 1])
for t, color in method_colors.items():
    sub = df[df["type"] == t]
    ax.scatter(sub["psnr_mean"], sub["privacy_score"], label=t, color=color, s=35, alpha=0.75)
ax.set_title("PSNR reference vs score")
ax.set_xlabel("PSNR dB")
ax.set_ylabel("score")
ax.grid(True)

ax = fig.add_subplot(gs[1, 2])
ax.hist(combo_df["privacy_score"], bins=18, color=GRAY, edgecolor="white")
ax.set_title("Combo score distribution")
ax.set_xlabel("score")
ax.set_ylabel("count")
ax.grid(axis="y")

fig.suptitle("12. Privacy Score Dashboard - MS-SSIM Based Score, PSNR/SSIM Reference", fontsize=16)
_save(fig, "12_dashboard_detailed.png")

print(f"상세 그래프 저장 완료: {GRAPH_DIR}")
